In [1]:
!pip install datasets==2.19.1 transformers evaluate seqeval -q
!pip install fsspec==2024.3.1 gcsfs==2024.3.1 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


# 1. Import Libraries


In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

# 2. Load Dataset


In [3]:
dataset = load_dataset("conll2003")

print(dataset)

print("\nSelected Dataset: CoNLL-2003")

label_names = dataset["train"].features["chunk_tags"].feature.names

print("\nChunk Labels:")
print(label_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Selected Dataset: CoNLL-2003

Chunk Labels:
['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


# 3. Load Tokenizer


In [4]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# 4. Tokenize + Align Labels


In [5]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]): # Changed from 'chunk_tags' to 'ner_tags', confirmed 'ner_tags' for conllpp
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

print(tokenized_dataset["train"][0].keys())

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

dict_keys(['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])


# 5. Model Setup

In [6]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
data_collator = DataCollatorForTokenClassification(tokenizer)


# 6. Evaluation Metric


In [8]:
metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):

    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=2)

    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


# 7. Training


In [9]:
training_args = TrainingArguments(
    output_dir="./bert_chunk_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.069878,0.057004,0.908713,0.919724,0.914185,0.983704
2,0.041372,0.052088,0.922911,0.938909,0.930842,0.986118
3,0.023937,0.050635,0.929416,0.939583,0.934472,0.986839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.07878893631918464, metrics={'train_runtime': 260.9973, 'train_samples_per_second': 161.392, 'train_steps_per_second': 10.092, 'total_flos': 510251380802730.0, 'train_loss': 0.07878893631918464, 'epoch': 3.0})

# 8. Evaluation


In [10]:
results = trainer.evaluate()

print("\nEvaluation Results:")
print(results)


Evaluation Results:
{'eval_loss': 0.05063549429178238, 'eval_precision': 0.9294156817046779, 'eval_recall': 0.9395826321104005, 'eval_f1': 0.9344715038915391, 'eval_accuracy': 0.9868385187492699, 'eval_runtime': 5.8874, 'eval_samples_per_second': 552.028, 'eval_steps_per_second': 34.65, 'epoch': 3.0}


# 9. Save Model


In [11]:
trainer.save_model("./final_chunk_model")
tokenizer.save_pretrained("./final_chunk_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_chunk_model/tokenizer_config.json',
 './final_chunk_model/tokenizer.json')

# 10. Inference


In [12]:
from transformers import pipeline

classifier = pipeline(
    "token-classification",
    model="./final_chunk_model",
    tokenizer="./final_chunk_model",
    aggregation_strategy="simple"
)

sentence = "John works at Google in California"

predictions = classifier(sentence)

print("\nInput Sentence:")
print(sentence)

print("\nPredictions:")
for pred in predictions:
    print(pred)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Input Sentence:
John works at Google in California

Predictions:
{'entity_group': 'ADJP', 'score': np.float32(0.99501824), 'word': 'john', 'start': 0, 'end': 4}
{'entity_group': 'ADVP', 'score': np.float32(0.9921733), 'word': 'google', 'start': 14, 'end': 20}
{'entity_group': 'CONJP', 'score': np.float32(0.9932596), 'word': 'california', 'start': 24, 'end': 34}


# Comparison


*   POS Tagging -> Identifies grammatical role of each word

*   Chunking -> Groups words into phrases like NP, VP, PP




# Report


1. POS Tagging labels words as noun, verb, adjective etc.
2. Chunking identifies phrases such as noun phrase and verb phrase.
3. Main challenge was aligning labels after subword tokenization.
4. DistilBERT gives strong performance with fast training.
5. Transformer models outperform traditional NLP methods.